# 환경설정

In [ ]:
# 인증

from google.colab import auth
auth.authenticate_user()
print("인증 완료")

In [ ]:
# 매직 로드

%load_ext google.cloud.bigquery

In [ ]:
# 시각화 설정

!pip install koreanize-matplotlib -q
import koreanize_matplotlib
import matplotlib.pyplot as plt

# mart_activation — AARRR 2단계 · 활성화(Activation) Mart

**AARRR 정의 (팀 합의)**
1. Acquisition(획득): 가입 (`mart_acquisition`에서 이미 완료)
2. **Activation(활성화)**: 첫 수신 + 힌트 열람 경험 + (40명 이상 학교, 4명 이상 반 학생 조건 충족) ← 이 노트북
3. Retention(유지): 재방문 (기간 추후 논의)
4. 핵심 가설: "첫 수신과 힌트 열람을 모두 경험한 유저는 그렇지 않은 유저에 비해 재방문율이 높을 것"

**목적**: `mart_acquisition`(가입 base)에 활성화 관련 컬럼을 이어 붙인다. `user_id`가 공통 키.

**활성화 관련 세부 지표 4가지 (팀 정의)**
1. 처음으로 받은 시간 — `first_receive_at` (첫 액션/메시지 수신 시점)
2. 수신 여부 — `has_first_receive` (유저가 메시지/초대 등을 정상적으로 받았는지)
3. 받은 개수 — `receive_count` (누적 수신 건수)
4. 힌트 열람 — `has_hint_opened`(한 번이라도 열람했는지) + `first_hint_opened_at`(시점) +
   `time_to_hint_open_hours`(첫 수신 후 힌트 열람까지 걸린 시간 — 핵심 기능을 얼마나 빨리 경험하는지)

**활성화(is_activated) 최종 조건 4가지 (전부 충족해야 TRUE)**
1. 첫 수신 경험 (`has_first_receive`)
2. 힌트 열람 경험 (`has_hint_opened`)
3. 학교 임계 규모 충족 — **가입 시점까지 누적된** 소속 학교 가입자 수 ≥ 40명
4. 반 임계 규모 충족 — **가입 시점까지 누적된** 소속 반 가입자 수 ≥ 4명

---

## 회의 결론 반영 (수정 사항)

| 논의 항목 | 결론 | 반영 내용 |
|---|---|---|
| 1. 임계 규모 기준이 "지금 시점"인지 "가입 시점"인지 | **가입 시점** | `mart_acquisition` 전체의 현재 누적 인원이 아니라, 윈도우 함수로 **각 유저가 가입한 순간까지 그 학교/반에 누적된 인원**을 계산해서 40명/4명 여부를 판단하도록 변경 (`school_signup_count_asof_signup`, `class_signup_count_asof_signup`) |
| 2. receive_count를 어디까지 볼지 | **누적 유지** | 기간 제한 없이 전체 기간 누적 카운트 그대로 사용 (변경 없음) |
| 3. time_to_hint_open_hours 활용 방안 | **구간 나열 후 관찰** | 임계값(예: "N시간 이내")을 아직 정하지 않고, `time_to_hint_open_bucket`으로 구간을 나눠 분포만 관찰. `is_activated` 판정에는 사용하지 않음 (힌트 열람 여부만 사용) |

⚠️ 임계 규모를 "가입 시점" 기준으로 바꾸면서, 같은 학교/반이라도 유저마다 `school_signup_count_asof_signup`
값이 달라집니다 (먼저 가입한 유저는 낮은 값, 나중에 가입한 유저는 높은 값). 이건 의도된 동작입니다 —
"그 유저가 활성화될 수 있었던 시점의 조건"을 보기 위함입니다.

**산출물**
- `mart.mart_activation` — 유저 grain (Acquisition 확장)
- `mart.mart_activation_summary` — 세그먼트별 활성화율 요약 (대시보드용)


## 1. stg_vote 컬럼 확인 (사용 전 재확인)

In [ ]:
%%bigquery df_vote_schema --project your-gcp-project

SELECT column_name, data_type
FROM `your-gcp-project.stage.INFORMATION_SCHEMA.COLUMNS`
WHERE table_name = 'stg_vote'
ORDER BY ordinal_position

In [ ]:
df_vote_schema
# 아래 쿼리는 receiver_user_id, is_hint_opened, voted_at 컬럼명을 가정합니다.
# 다르게 나오면 2번부터 해당 컬럼명만 바꿔주세요.

## 2. 학교 · 반 임계 규모 사전 확인 (가입 시점 기준)

`mart_acquisition`을 기준으로, 각 유저가 **가입한 시점까지** 그 학교/반에 누적된 가입자 수를 윈도우 함수로 계산합니다.
(회의 결론 1번 반영 — "지금 시점 누적"이 아니라 "그 유저 가입 시점 누적")


In [ ]:
%%bigquery df_threshold_check --project your-gcp-project

WITH asof AS (
  SELECT
    user_id,
    school_id,
    group_id,
    COUNT(*) OVER (
      PARTITION BY school_id
      ORDER BY signup_at
      RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS school_signup_count_asof_signup,
    COUNT(*) OVER (
      PARTITION BY group_id
      ORDER BY signup_at
      RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS class_signup_count_asof_signup
  FROM `your-gcp-project.mart.mart_acquisition`
)
SELECT
  COUNT(*) AS total_users,
  COUNTIF(school_signup_count_asof_signup >= 40) AS users_school_threshold_met_at_signup,
  ROUND(COUNTIF(school_signup_count_asof_signup >= 40) / COUNT(*) * 100, 1) AS school_threshold_met_pct,
  COUNTIF(class_signup_count_asof_signup >= 4) AS users_class_threshold_met_at_signup,
  ROUND(COUNTIF(class_signup_count_asof_signup >= 4) / COUNT(*) * 100, 1) AS class_threshold_met_pct
FROM asof

In [ ]:
df_threshold_check
# 가입 시점 기준이라, "현재 누적 기준(이전 EDA의 below_4_pct 등)"과 숫자가 다를 수 있습니다.
# 이건 정상입니다 — 관점이 다른 지표입니다 (현재 상태 vs 가입 당시 상태).

## 3. 최종 정제 미리보기

In [ ]:
%%bigquery df_activation_preview --project your-gcp-project

WITH asof_threshold AS (
  SELECT
    user_id,
    COUNT(*) OVER (
      PARTITION BY school_id
      ORDER BY signup_at
      RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS school_signup_count_asof_signup,
    COUNT(*) OVER (
      PARTITION BY group_id
      ORDER BY signup_at
      RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS class_signup_count_asof_signup
  FROM `your-gcp-project.mart.mart_acquisition`
),
first_receive AS (
  SELECT
    receiver_user_id AS user_id,
    MIN(voted_at) AS first_receive_at,
    COUNT(*) AS receive_count
  FROM `your-gcp-project.stage.stg_vote`
  GROUP BY receiver_user_id
),
first_hint AS (
  SELECT
    receiver_user_id AS user_id,
    MIN(voted_at) AS first_hint_opened_at
  FROM `your-gcp-project.stage.stg_vote`
  WHERE is_hint_opened = TRUE
  GROUP BY receiver_user_id
)
SELECT
  a.user_id,
  a.signup_date,
  a.school_id,
  a.school_type,
  a.group_id,
  a.grade,
  a.class_num,

  fr.first_receive_at,
  fr.first_receive_at IS NOT NULL AS has_first_receive,
  COALESCE(fr.receive_count, 0) AS receive_count,

  fh.first_hint_opened_at,
  fh.first_hint_opened_at IS NOT NULL AS has_hint_opened,
  DATETIME_DIFF(fh.first_hint_opened_at, fr.first_receive_at, HOUR) AS time_to_hint_open_hours,

  CASE
    WHEN fh.first_hint_opened_at IS NULL THEN '힌트 미열람'
    WHEN DATETIME_DIFF(fh.first_hint_opened_at, fr.first_receive_at, HOUR) < 1 THEN '1시간 이내'
    WHEN DATETIME_DIFF(fh.first_hint_opened_at, fr.first_receive_at, HOUR) < 6 THEN '1~6시간'
    WHEN DATETIME_DIFF(fh.first_hint_opened_at, fr.first_receive_at, HOUR) < 24 THEN '6~24시간'
    WHEN DATETIME_DIFF(fh.first_hint_opened_at, fr.first_receive_at, HOUR) < 72 THEN '1~3일'
    ELSE '3일 이상'
  END AS time_to_hint_open_bucket,

  ath.school_signup_count_asof_signup,
  ath.school_signup_count_asof_signup >= 40 AS meets_school_threshold,

  ath.class_signup_count_asof_signup,
  ath.class_signup_count_asof_signup >= 4 AS meets_class_threshold,

  (fr.first_receive_at IS NOT NULL)
    AND (fh.first_hint_opened_at IS NOT NULL)
    AND (ath.school_signup_count_asof_signup >= 40)
    AND (ath.class_signup_count_asof_signup >= 4) AS is_activated

FROM `your-gcp-project.mart.mart_acquisition` a
LEFT JOIN asof_threshold ath ON a.user_id = ath.user_id
LEFT JOIN first_receive fr ON a.user_id = fr.user_id
LEFT JOIN first_hint fh ON a.user_id = fh.user_id
LIMIT 30;

In [ ]:
df_activation_preview.head(5)

## 4. 최종 Mart 생성

In [ ]:
%%bigquery --project your-gcp-project
CREATE OR REPLACE TABLE `your-gcp-project.mart.mart_activation` AS
WITH asof_threshold AS (
  -- 회의 결론 1: 임계 규모는 "지금 시점 누적"이 아니라 "가입 시점까지 누적된 인원" 기준
  SELECT
    user_id,
    COUNT(*) OVER (
      PARTITION BY school_id
      ORDER BY signup_at
      RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS school_signup_count_asof_signup,
    COUNT(*) OVER (
      PARTITION BY group_id
      ORDER BY signup_at
      RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS class_signup_count_asof_signup
  FROM `your-gcp-project.mart.mart_acquisition`
),
first_receive AS (
  -- 회의 결론 2: receive_count는 누적 유지 (기간 제한 없음)
  SELECT
    receiver_user_id AS user_id,
    MIN(voted_at) AS first_receive_at,
    COUNT(*) AS receive_count
  FROM `your-gcp-project.stage.stg_vote`
  GROUP BY receiver_user_id
),
first_hint AS (
  -- 4. 한 번이라도 힌트 열람했는지 + 시점
  SELECT
    receiver_user_id AS user_id,
    MIN(voted_at) AS first_hint_opened_at
  FROM `your-gcp-project.stage.stg_vote`
  WHERE is_hint_opened = TRUE
  GROUP BY receiver_user_id
)
SELECT
  a.user_id,
  a.signup_date,
  a.signup_at,
  a.school_id,
  a.school_type,
  a.group_id,
  a.grade,
  a.class_num,
  a.gender,

  -- 1·2·3. 수신 관련
  fr.first_receive_at,
  fr.first_receive_at IS NOT NULL AS has_first_receive,
  COALESCE(fr.receive_count, 0) AS receive_count,

  -- 4. 힌트 열람 관련 (여부 · 시점 · 첫 수신 후 걸린 시간)
  fh.first_hint_opened_at,
  fh.first_hint_opened_at IS NOT NULL AS has_hint_opened,
  DATETIME_DIFF(fh.first_hint_opened_at, fr.first_receive_at, HOUR) AS time_to_hint_open_hours,

  -- 회의 결론 3: 임계값 미정, 구간으로만 관찰 (is_activated 판정에는 미사용)
  CASE
    WHEN fh.first_hint_opened_at IS NULL THEN '힌트 미열람'
    WHEN DATETIME_DIFF(fh.first_hint_opened_at, fr.first_receive_at, HOUR) < 1 THEN '1시간 이내'
    WHEN DATETIME_DIFF(fh.first_hint_opened_at, fr.first_receive_at, HOUR) < 6 THEN '1~6시간'
    WHEN DATETIME_DIFF(fh.first_hint_opened_at, fr.first_receive_at, HOUR) < 24 THEN '6~24시간'
    WHEN DATETIME_DIFF(fh.first_hint_opened_at, fr.first_receive_at, HOUR) < 72 THEN '1~3일'
    ELSE '3일 이상'
  END AS time_to_hint_open_bucket,

  -- 회의 결론 1: 가입 시점 기준 임계 규모
  ath.school_signup_count_asof_signup,
  ath.school_signup_count_asof_signup >= 40 AS meets_school_threshold,

  ath.class_signup_count_asof_signup,
  ath.class_signup_count_asof_signup >= 4 AS meets_class_threshold,

  (fr.first_receive_at IS NOT NULL)
    AND (fh.first_hint_opened_at IS NOT NULL)
    AND (ath.school_signup_count_asof_signup >= 40)
    AND (ath.class_signup_count_asof_signup >= 4) AS is_activated

FROM `your-gcp-project.mart.mart_acquisition` a
LEFT JOIN asof_threshold ath ON a.user_id = ath.user_id
LEFT JOIN first_receive fr ON a.user_id = fr.user_id
LEFT JOIN first_hint fh ON a.user_id = fh.user_id;

### 4-1. 생성 후 검증

In [ ]:
%%bigquery --project your-gcp-project
SELECT
  COUNT(*) AS total_rows,
  COUNT(DISTINCT user_id) AS distinct_user_id,        -- total_rows와 같아야 함
  COUNTIF(has_first_receive) AS first_receive_count,
  ROUND(AVG(receive_count), 1) AS avg_receive_count,
  COUNTIF(has_hint_opened) AS hint_opened_count,
  ROUND(AVG(time_to_hint_open_hours), 1) AS avg_time_to_hint_open_hours,  -- NULL은 자동 제외되고 평균 계산됨
  COUNTIF(meets_school_threshold) AS school_ok_count_asof_signup,
  COUNTIF(meets_class_threshold) AS class_ok_count_asof_signup,
  COUNTIF(is_activated) AS activated_count,
  ROUND(COUNTIF(is_activated) / COUNT(*) * 100, 1) AS activation_rate_pct
FROM `your-gcp-project.mart.mart_activation`;

## 5. mart_activation_summary — 세그먼트별 활성화율 (대시보드용)

In [ ]:
%%bigquery --project your-gcp-project
CREATE OR REPLACE TABLE `your-gcp-project.mart.mart_activation_summary` AS
SELECT
  school_type,
  meets_school_threshold,
  meets_class_threshold,
  COUNT(*) AS total_users,
  COUNTIF(is_activated) AS activated_users,
  ROUND(COUNTIF(is_activated) / COUNT(*) * 100, 1) AS activation_rate_pct
FROM `your-gcp-project.mart.mart_activation`
GROUP BY school_type, meets_school_threshold, meets_class_threshold
ORDER BY school_type, meets_school_threshold, meets_class_threshold;

In [ ]:
%%bigquery df_summary_check --project your-gcp-project
SELECT * FROM `your-gcp-project.mart.mart_activation_summary`
ORDER BY school_type, meets_school_threshold, meets_class_threshold

In [ ]:
df_summary_check

### 5-1. 임계 규모 충족 여부에 따른 활성화율 비교 시각화

In [ ]:
plt.figure(figsize=(8, 4))
grouped = df_summary_check.groupby(['meets_school_threshold', 'meets_class_threshold'])['activation_rate_pct'].mean()
grouped.plot(kind='bar')
plt.ylabel('평균 활성화율 (%)')
plt.title('학교/반 임계 규모 충족 여부별 활성화율')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### 5-2. time_to_hint_open_hours 구간별 분포 관찰 (회의 결론 3 반영)

아직 "N시간 이내면 빠른 편"이라는 임계값을 정하지 않았으므로, `is_activated` 판정에는 반영하지 않고
구간별 분포만 관찰합니다. 이 결과를 보고 팀에서 기준을 논의하면 됩니다.

In [ ]:
%%bigquery df_hint_open_bucket --project your-gcp-project

SELECT
  time_to_hint_open_bucket,
  COUNT(*) AS user_count,
  ROUND(COUNT(*) / SUM(COUNT(*)) OVER () * 100, 1) AS pct
FROM `your-gcp-project.mart.mart_activation`
GROUP BY time_to_hint_open_bucket
ORDER BY
  CASE time_to_hint_open_bucket
    WHEN '1시간 이내' THEN 1
    WHEN '1~6시간' THEN 2
    WHEN '6~24시간' THEN 3
    WHEN '1~3일' THEN 4
    WHEN '3일 이상' THEN 5
    WHEN '힌트 미열람' THEN 6
  END

In [ ]:
df_hint_open_bucket

In [ ]:
plt.figure(figsize=(8, 4))
order = ['1시간 이내', '1~6시간', '6~24시간', '1~3일', '3일 이상', '힌트 미열람']
plot_df = df_hint_open_bucket.set_index('time_to_hint_open_bucket').reindex(order)
plt.bar(plot_df.index, plot_df['user_count'])
plt.ylabel('유저 수')
plt.title('첫 수신 후 힌트 열람까지 걸린 시간 (구간별 분포)')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

## 다음 단계

- `meets_school_threshold`/`meets_class_threshold`가 FALSE인 유저는 애초에 활성화가 구조적으로
  불가능한 상태(질문 세트 자체가 실행 안 됨)이므로, 축B(학교 네트워크)의 "임계 규모가 실제로
  존재하는가" 질문에 대한 핵심 근거가 됩니다. 5번 시각화 결과를 꼭 확인하세요.
- 이제 `mart_activation`의 `user_id`를 기준으로 Retention Mart(재방문)를 이어 붙이면,
  핵심 가설("첫 수신+힌트 열람 경험자의 재방문율이 더 높다")을 바로 검증할 수 있습니다.
  다만 재방문 기간(며칠 기준)은 아직 팀 합의 전이니, 그 부분부터 확정하고 진행하시는 걸 추천드려요.


## 부록: 이상치 조사 — 5월 가입 스파이크

3번(최종 정제 미리보기) 단계에서 첫 힌트 열람 추이를 일별로 그려보다가 5월 구간에서 튀는 지점을
발견해서, 진짜 트렌드인지 데이터 이상치(일괄 등록 등)인지 확인한 기록입니다.
**최종 `mart_activation` 생성 로직에는 영향을 주지 않으며, 참고용 기록으로 남겨둡니다.**


In [ ]:
%%bigquery df_first_hint_daily_simple --project your-gcp-project

WITH first_hint AS (
  SELECT
    receiver_user_id AS user_id,
    MIN(voted_at) AS first_hint_opened_at
  FROM `your-gcp-project.stage.stg_vote`
  WHERE is_hint_opened = TRUE
  GROUP BY receiver_user_id
)
SELECT
  DATE(first_hint_opened_at) AS hint_open_date,
  COUNT(*) AS hint_open_count
FROM first_hint
GROUP BY hint_open_date
ORDER BY hint_open_date

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(df_first_hint_daily_simple['hint_open_date'], df_first_hint_daily_simple['hint_open_count'])
plt.xlabel('힌트 최초 열람일')
plt.ylabel('유저 수')
plt.title('일별 첫 힌트 열람 추이 (전체)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### 5월 스파이크가 특정 지역에 몰려있는지 확인

In [ ]:
%%bigquery df_may_spike_region --project your-gcp-project

SELECT
  s.address,
  COUNT(*) AS signup_count
FROM `your-gcp-project.mart.mart_acquisition` a
JOIN `your-gcp-project.stage.stg_school` s
  ON a.school_id = s.school_id
WHERE a.signup_date BETWEEN '2023-05-01' AND '2023-05-31'
GROUP BY s.address
ORDER BY signup_count DESC
LIMIT 20

In [ ]:
df_may_spike_region

### 일괄 등록인지 확인 (학교 단위)

In [ ]:
%%bigquery df_may_spike_by_school --project your-gcp-project

SELECT
  a.school_id,
  s.school_type,
  s.address,
  COUNT(*) AS may_signup_count
FROM `your-gcp-project.mart.mart_acquisition` a
JOIN `your-gcp-project.stage.stg_school` s
  ON a.school_id = s.school_id
WHERE a.signup_date BETWEEN '2023-05-01' AND '2023-05-31'
GROUP BY a.school_id, s.school_type, s.address
ORDER BY may_signup_count DESC
LIMIT 20

In [ ]:
df_may_spike_by_school

### 같은 학교 안에서 가입 시각이 짧은 간격에 몰려있는지 확인

In [ ]:
%%bigquery df_bulk_signup_check --project your-gcp-project

SELECT
  a.school_id,
  DATE(a.signup_at) AS signup_day,
  EXTRACT(HOUR FROM a.signup_at) AS signup_hour,
  COUNT(*) AS signup_count_in_hour
FROM `your-gcp-project.mart.mart_acquisition` a
WHERE a.signup_date BETWEEN '2023-05-01' AND '2023-05-31'
GROUP BY a.school_id, signup_day, signup_hour
HAVING COUNT(*) >= 10
ORDER BY signup_day ASC
LIMIT 30

In [ ]:
df_bulk_signup_check

### 6월 이후 신규 학교 vs 기존 학교 비교

In [ ]:
%%bigquery df_new_vs_existing_school --project your-gcp-project

WITH school_first_signup AS (
  SELECT school_id, MIN(signup_date) AS first_signup_date
  FROM `your-gcp-project.mart.mart_acquisition`
  GROUP BY school_id
)
SELECT
  FORMAT_DATE('%Y-%m', a.signup_date) AS signup_month,
  COUNT(*) AS total_signups,
  COUNT(DISTINCT CASE WHEN FORMAT_DATE('%Y-%m', sf.first_signup_date) = FORMAT_DATE('%Y-%m', a.signup_date) THEN a.school_id END) AS new_schools_this_month,
  COUNT(DISTINCT CASE WHEN FORMAT_DATE('%Y-%m', sf.first_signup_date) != FORMAT_DATE('%Y-%m', a.signup_date) THEN a.school_id END) AS returning_schools_this_month
FROM `your-gcp-project.mart.mart_acquisition` a
JOIN school_first_signup sf ON a.school_id = sf.school_id
GROUP BY signup_month
ORDER BY signup_month

In [ ]:
df_new_vs_existing_school

### 5월 가입만으로 전교생 대비 몇 %를 차지하는지

In [ ]:
%%bigquery df_bulk_vs_threshold --project your-gcp-project

SELECT
  s.school_id,
  s.student_count,
  a_may.may_signup_count,
  ROUND(a_may.may_signup_count / NULLIF(s.student_count, 0) * 100, 1) AS may_alone_pct
FROM `your-gcp-project.stage.stg_school` s
JOIN (
  SELECT school_id, COUNT(*) AS may_signup_count
  FROM `your-gcp-project.mart.mart_acquisition`
  WHERE signup_date BETWEEN '2023-05-01' AND '2023-05-31'
  GROUP BY school_id
) a_may
ON s.school_id = a_may.school_id
ORDER BY may_alone_pct DESC
LIMIT 20

In [ ]:
df_bulk_vs_threshold

### 월 경계(5월 말 ~ 6월 초) 일별 가입 추이 확인

In [ ]:
%%bigquery df_month_boundary_check --project your-gcp-project

SELECT
  DATE(signup_at) AS signup_day,
  COUNT(*) AS signup_count,
  COUNT(DISTINCT school_id) AS distinct_school_count
FROM `your-gcp-project.mart.mart_acquisition`
WHERE signup_date BETWEEN '2023-05-28' AND '2023-06-05'
GROUP BY signup_day
ORDER BY signup_day

In [ ]:
df_month_boundary_check

### 6월 주간 가입 추이 확인

In [ ]:
%%bigquery df_june_weekly --project your-gcp-project

SELECT
  signup_date,
  COUNT(*) AS signup_count
FROM `your-gcp-project.mart.mart_acquisition`
WHERE signup_date BETWEEN '2023-06-01' AND '2023-06-30'
GROUP BY signup_date
ORDER BY signup_date

In [ ]:
df_june_weekly